# Lesson 7 - Creating an A2A Sequential Chain Agent with ADK

Now that you have two active agents (Policy Agent and Research Agent), you will orchestrate them. In this lesson, you will use Google ADK's `SequentialAgent` to create a workflow where a user's query is processed by the Research Agent first, and then the Policy Agent. You will use `RemoteA2aAgent` (which acts as A2A Client) to connect to the servers you started in previous lessons.

![Sequential Workflow](sequential.png)

## 7.1. Start the A2A Servers

First, ensure that your `PolicyAgent` server is running.
- Open Terminal 1 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_policy_agent.py` (you don't need to go back to the previous lesson).

In [1]:
import os

from IPython.display import IFrame

url = os.environ.get("DLAI_LOCAL_URL").format(port=8888)
# Terminal 1: uv run a2a_policy_agent.py
IFrame(f"{url}terminals/1", width=800, height=200)

Second, ensure that your Research Agent is running.
- Open Terminal 2 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_research_agent.py` (you don't need to go back to the previous lesson).

In [2]:
# Terminal 2: uv run a2a_research_agent.py
IFrame(f"{url}terminals/2", width=800, height=200)

## 7.2. Define the Sequential Workflow

Here you will:
1.  Define two `RemoteA2aAgent` instances. These act as A2A clients that know how to communicate with your running servers via A2A.
2.  Create a `SequentialAgent` named `root_agent`. This agent contains the logic to route the workflow through the sub-agents.
3.  Use an `InMemoryRunner` to execute the flow with a specific prompt.


In [3]:
import os

from IPython.display import Markdown, display
from dotenv import load_dotenv
from google.adk.agents import SequentialAgent
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
)
from google.adk.runners import InMemoryRunner

import logging
import warnings

logging.disable(level=logging.WARNING)
warnings.filterwarnings("ignore")

In [4]:
load_dotenv()
host = os.environ.get("AGENT_HOST")
policy_port = os.environ.get("POLICY_AGENT_PORT")
research_port = os.environ.get("RESEARCH_AGENT_PORT")

In [5]:
policy_agent = RemoteA2aAgent(
    name="policy_agent",
    agent_card=f"http://{host}:{policy_port}",
)
print("\tℹ️", f"{policy_agent.name} initialized")

	ℹ️ policy_agent initialized


In [6]:
health_research_agent = RemoteA2aAgent(
    name="health_research_agent",
    agent_card=f"http://{host}:{research_port}",
)
print("\tℹ️", f"{health_research_agent.name} initialized")

	ℹ️ health_research_agent initialized


In [7]:
root_agent = SequentialAgent(
    name="root_agent",
    description="Healthcare Routing Agent",
    sub_agents=[
        health_research_agent,
        policy_agent,
    ],
)
print("\tℹ️", f"{root_agent.name} initialized")

	ℹ️ root_agent initialized


## 7.3. Run the Sequential Chain

The `InMemoryRunner` executes the agent. The prompt will trigger the Research Agent to find general info, and then (sequentially) the Policy Agent to check coverage details.

In [8]:
prompt = "How can I get mental health therapy?"

**Note:** It takes a few seconds for the output to display.

In [9]:
print("Running Healthcare Workflow Agent")

runner = InMemoryRunner(root_agent)

for event in await runner.run_debug(prompt, quiet=True):
    if event.is_final_response() and event.content:
        display(Markdown(event.content.parts[0].text))

Running Healthcare Workflow Agent


Taking the step to get mental health therapy is a courageous and important decision for your well-being. The process of finding the right therapist can sometimes feel overwhelming, but it is highly manageable when broken down into clear steps. Here is a comprehensive guide on how you can access mental health counseling:

### 1. Identify Your Needs and Goals
Before you start searching, take a moment to reflect on what you want to achieve in therapy and what type of professional you would be most comfortable with. 
* **Your Goals:** What is bringing you to therapy right now? Consider the specific issues you want to address, such as anxiety, depression, stress, relationship challenges, or coping with past trauma.
* **Therapist Preferences:** Think about whether you would prefer a therapist of a specific gender, age, cultural background, or religion. Some therapists specialize in working with specific populations, such as LGBTQ+ or BIPOC individuals.
* **Format:** Decide if you are more comfortable with in-person sessions, virtual online telehealth visits, individual therapy, or group counseling.
* **Therapy Style:** Different therapists use different methods. Some use structured models like Cognitive Behavioral Therapy (CBT) or Dialectical Behavior Therapy (DBT), while others might use trauma-informed care, psychodynamics, or integrative methods like art therapy and mindfulness. It is completely okay if you don't know exactly what modality you need yet.

### 2. Determine Your Payment Options
Therapy costs can vary, but there are multiple ways to cover or reduce the expense:
* **Private Insurance:** Check with your health insurance provider to understand your behavioral health benefits. Find out if you need a referral from your primary care physician, what your copayment is, and if you have out-of-network benefits.
* **Medicaid and Medicare:** Many state Medicaid and Medicare programs cover mental health services, including individual therapy, group counseling, and telehealth, often with zero or low copayments. 
* **Employee Assistance Programs (EAPs):** Check if your employer offers an EAP. EAPs often provide a certain number of free, confidential therapy sessions.
* **Sliding Scale:** If you are paying out-of-pocket and are underinsured or uninsured, many therapists offer a "sliding scale" fee structure. This means the fee is reduced based on your income and what you can afford. 
* **Community Mental Health Centers:** These centers offer reduced rates and provide comprehensive services like counseling and psychiatric care for low-income individuals. 

### 3. Search for a Therapist
Once you know what you are looking for and how you will pay, you can start looking for a provider:
* **Online Directories:** Websites like Psychology Today, TherapyDen, and Zocdoc allow you to easily filter therapists by location, specialty, insurance accepted, gender, and language. State psychological associations also provide member listings.
* **Personal Referrals:** Reach out to trusted friends or family members who are in therapy and ask if they recommend their therapist. 
* **Medical Referrals:** Your Primary Care Provider (PCP) can be a great resource for a list of local therapist recommendations.
* **Veteran Resources:** If you are a veteran, you can contact the VA Mental Health Hotline or your local VA office to schedule appointments with VA counselors or community-based veteran therapists. 

### 4. Ask for a Consultation
Once you have narrowed down a list of potential therapists, reach out to them via phone, email, or their website. Many therapists offer a complimentary 15-minute introductory consultation call. During this initial contact, you can ask:
* Are you licensed and in good standing to provide mental health services?
* Do you have experience treating my specific concerns?
* What is your approach or treatment philosophy?
* Do you accept my insurance or offer a sliding scale?
* Are you currently taking new patients, and what is your availability?

### 5. Evaluate the Fit
The relationship and connection between you and your therapist are among the biggest predictors of success in therapy. Go into your first few sessions with an open mind, share your history honestly, and discuss your goals. A good therapist will provide a safe, non-judgmental space, validate your experiences, and work with you to set realistic steps toward mental wellness. 

Give it two to three sessions to determine if they are the right match. If you don't feel a connection, don't feel understood, or don't feel comfortable being vulnerable, it is completely normal and acceptable to try working with someone else. 

***Note on Urgent Help:*** *If you are experiencing a mental health emergency, psychological crisis, or severe distress, please seek immediate help. You can call or text the Suicide & Crisis Lifeline at **988**, or go to an urgent walk-in crisis receiving center or emergency room.*

Based on the insurance document provided, here's how you can get mental health therapy coverage under this plan:

## Coverage for Mental Health Services

This plan covers **outpatient mental health, behavioral health, and substance abuse services** with the following costs after your deductible is met:

### **In-Network Providers:**
- **Office Visit:** 10% coinsurance
- **Other Outpatient Services:** 10% coinsurance

### **Out-of-Network Providers:**
- **Office Visit:** 30% coinsurance
- **Other Outpatient Services:** 30% coinsurance

### **Inpatient Services** (if needed):
- **In-Network:** 10% coinsurance
- **Out-of-Network:** 30% coinsurance

## Steps to Access Mental Health Therapy:

1. **Find an in-network provider** to minimize your costs. You can use:
   - **includedhealth.com/google** or call **(855) 431-5540** for a list of network providers

2. **You do not need a referral** to see a mental health specialist - you can choose any provider covered by the plan

3. **Remember your deductible:** You must meet your \\$1,700 individual (or \\$3,400 family) deductible before coinsurance costs apply

4. **Your out-of-pocket limit** is \\$2,600 per individual (or \\$5,200 per family) for in-network providers

For more detailed information about specific mental health providers and coverage details, consult your complete plan document at go/benefitdocuments or contact the number above.

## 7.4. Resources

- [Google ADK Sequential Agents](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)
- [Equivalent notebook in the course repo](https://github.com/holtskinner/A2AWalkthrough/blob/main/5_ADKSequentialAgent.ipynb)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>
